# Data Engineer Agent Tests

Use this notebook to validate the Data Engineer agent end to end: question -> SQL -> pandas DataFrame.

Before running the notebook, make sure:
- `data/orders.db` exists
- `GOOGLE_API_KEY` is available in the environment used by the notebook kernel

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path("..").resolve()
dotenv_path = project_root / ".env"

if not project_root.exists():
    raise FileNotFoundError(
        "Could not find the AskData project root from the current notebook working directory."
    )

if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

load_dotenv(dotenv_path=dotenv_path, override=True)

print(f"Project root: {project_root}")
print(f"Loaded environment from: {dotenv_path}")
print(f"GOOGLE_API_KEY loaded: {bool(os.getenv('GOOGLE_API_KEY'))}")

project_root

Project root: /Users/zac_burns/Documents/Personal/AskData
Loaded environment from: /Users/zac_burns/Documents/Personal/AskData/.env
GOOGLE_API_KEY loaded: True


PosixPath('/Users/zac_burns/Documents/Personal/AskData')

In [2]:
import pandas as pd

from askdata.agent import DataEngineerAgent
from askdata.database import OrdersDatabase
from askdata.validator import SqlValidator

In [3]:
database = OrdersDatabase(project_root / "data" / "orders.db")
database.ensure_exists()

print(f"Database path: {database.db_path}")
print("Available tables:", database.list_tables())
print()

schema = database.get_schema_summary()

for line in schema.split("\n"):
    if line.strip():
        # Split table name from columns
        table, columns = line.split(": ", 1)
        print(f"\n{table}")  # e.g. "- orders"
        print("-" * 40)
        for col in columns.split(", "):
            print(f"  • {col}")  # e.g. "order_id TEXT"

Database path: /Users/zac_burns/Documents/Personal/AskData/data/orders.db
Available tables: ['orders']


- orders
----------------------------------------
  • order_id TEXT
  • customer_id TEXT
  • order_status TEXT
  • order_purchase_timestamp TEXT
  • order_approved_at TEXT
  • order_delivered_carrier_date TEXT
  • order_delivered_customer_date TEXT
  • order_estimated_delivery_date TEXT


In [5]:
load_dotenv(override=True)

if not os.getenv("GOOGLE_API_KEY"):
    raise EnvironmentError(
        "GOOGLE_API_KEY is not set in the notebook environment or .env file."
    )

agent = DataEngineerAgent(database=database, validator=SqlValidator(database))
agent

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
question = "How many orders do we have by status? Order by the most frequent"
result = agent.run(question)

print("Question:")
print(result.question)
print()
print("Generated SQL:")
print(result.sql)
print()
result.dataframe

Question:
How many orders do we have by status? Order by the most frequent

Generated SQL:
SELECT order_status, COUNT(order_id) AS order_count FROM orders GROUP BY order_status ORDER BY order_count DESC



,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [10]:
def ask(question: str) -> pd.DataFrame:
    result = agent.run(question)
    print("Generated SQL:")
    print(result.sql)
    print()

    return result.dataframe

In [9]:
ask("Which order statuses have the longest average delivery times?")

Generated SQL:
SELECT order_status, AVG(JULIANDAY(order_delivered_customer_date) - JULIANDAY(order_purchase_timestamp)) AS average_delivery_time_days FROM orders WHERE order_delivered_customer_date IS NOT NULL AND order_purchase_timestamp IS NOT NULL GROUP BY order_status ORDER BY average_delivery_time_days DESC



,order_status,average_delivery_time_days
0,canceled,20.360006
1,delivered,12.558217


,order_status,average_delivery_time_days
0,canceled,20.360006
1,delivered,12.558217
